[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bulentsoykan/ABM-particle-filter-notebooks/blob/main/notebook_02_pf_1d_animated.ipynb)

# Module 2: The Particle Filter — Core Concepts

**Learning objectives**
1. Understand the three-step PF cycle: Predict → Reweight → Resample
2. Visualize particles as a probability distribution over possible states
3. See how the filter tracks a nonlinear moving object step-by-step

> *"A particle filter is a brute force Bayesian state estimation method.
> The goal is to estimate a posterior distribution, i.e. the probability that the
> system is in a particular state conditioned on the observed data."*
> — Malleson et al. (2020), Section 2.8

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color':      '#8b949e', 'ytick.color':     '#8b949e',
    'text.color':       '#e6edf3', 'grid.color':      '#21262d',
    'grid.alpha': 0.5,  'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'axes.grid': True,
    'font.size': 11,
})
os.makedirs('figures', exist_ok=True)
print("Libraries loaded.")

## The PF Cycle Diagram

Before animating, let's build intuition with a static diagram.
The filter alternates between two phases:

- **Predict** — run the model forward; particles spread out (uncertainty grows)
- **Update** — compare to observation; reweight then resample (uncertainty shrinks)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('The Particle Filter Cycle (one iteration)', fontsize=14, fontweight='bold')

np.random.seed(7)
N_DEMO = 30
x_true = 5.0

titles = ['1. PRIOR\n(before this step)',
          '2. PREDICT\n(propagate forward)',
          '3. REWEIGHT\n(score vs observation)',
          '4. RESAMPLE\n(survival of the fittest)']
colors = ['#58a6ff', '#3fb950', '#f0883e', '#f9826c']

# 1. Prior: particles spread around some previous estimate
prior_x = np.random.normal(3.5, 1.5, N_DEMO)
prior_y = np.zeros(N_DEMO)

# 2. Predict: add motion noise
pred_x = prior_x + np.random.normal(1.2, 0.8, N_DEMO)
pred_y = np.zeros(N_DEMO)

# 3. Reweight: weights proportional to closeness to observation
dists   = np.abs(pred_x - x_true) + 0.01
weights = 1.0 / dists
weights /= weights.sum()

# 4. Resample: draw new particles according to weights
idxs      = np.random.choice(N_DEMO, size=N_DEMO, p=weights)
resamp_x  = pred_x[idxs]

for ax, title, col in zip(axes, titles, colors):
    ax.set_title(title, fontsize=11, color=col, fontweight='bold')
    ax.set_xlim(-2, 12)
    ax.set_ylim(-0.5, 1.5)
    ax.set_yticks([])
    ax.set_xlabel('State space (position)', fontsize=10)
    ax.axvline(x_true, color='#f9826c', lw=2.5, zorder=5, label='True state')

axes[0].scatter(prior_x, prior_y + 0.5, s=80, color='#58a6ff', alpha=0.7, zorder=3)
axes[0].text(x_true + 0.2, 1.2, 'TRUE\nSTATE', color='#f9826c', fontsize=9, ha='left')

axes[1].scatter(pred_x, pred_y + 0.5, s=80, color='#3fb950', alpha=0.7, zorder=3)
for px, ox in zip(prior_x[:8], pred_x[:8]):
    axes[1].annotate('', xy=(px + (ox-px), 0.5), xytext=(px, 0.5),
                     arrowprops=dict(arrowstyle='->', color='#8b949e', lw=1.2))

axes[2].scatter(pred_x, pred_y + 0.5,
                s=np.clip(weights * N_DEMO * 300, 10, 500),
                color='#f0883e', alpha=0.8, zorder=3)
axes[2].text(0.02, 0.92, 'Size = weight\n(close = heavy)',
             transform=axes[2].transAxes, fontsize=9, color='#f0883e')

axes[3].scatter(resamp_x, np.zeros(N_DEMO) + 0.5, s=80,
                color='#f9826c', alpha=0.7, zorder=3)
axes[3].text(0.02, 0.92, 'Duplicated heavy\nparticles, removed\nlight ones',
             transform=axes[3].transAxes, fontsize=9, color='#f9826c')

for ax in axes:
    ax.axvline(x_true, color='#f9826c', lw=2, ls='--', alpha=0.6, zorder=1)

plt.tight_layout()
plt.savefig('figures/fig_02a_pf_cycle_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"The filter estimate after resampling: {np.mean(resamp_x):.2f} | True state: {x_true}")

## Particle Filter Functions

The three core operations in pure NumPy.  These are the building blocks used in
all later notebooks.


In [ ]:
# ── Predict ───────────────────────────────────────────────────────────────────
def pf_predict(particles, motion_std=0.5):
    '''Propagate particles forward by adding Gaussian motion noise.

    In a real ABM this would be: 'run each particle model one step forward'.
    Here we just add random noise to simulate uncertain dynamics.
    '''
    return particles + np.random.randn(*particles.shape) * motion_std

# ── Reweight ──────────────────────────────────────────────────────────────────
def pf_reweight(particles, observation, sensor_std=0.5):
    '''Compute importance weights from distance to observation.

    Weight = 1 / (distance + epsilon)
    Particles close to the observation get high weight.
    Sensor noise is added to model imperfect measurements.
    '''
    noisy_obs = observation + np.random.randn(*observation.shape) * sensor_std
    dists     = np.linalg.norm(particles - noisy_obs, axis=1)
    weights   = 1.0 / (dists + 1e-300)
    return weights / weights.sum()

# ── Resample (systematic) ─────────────────────────────────────────────────────
def systematic_resample(weights):
    '''Systematic resampling — O(N), low variance, preferred for PF.

    Algorithm (Eq. 5-6 in Malleson et al. 2020):
      1. Draw one random U ~ Uniform[0, 1/N]
      2. Space N points evenly: U, U+1/N, U+2/N, ..., U+(N-1)/N
      3. Walk the CDF; each particle covers a slice proportional to its weight
         -> heavy particles get sampled multiple times, light ones get dropped
    '''
    N   = len(weights)
    w   = np.asarray(weights, dtype=float)
    w   = w / w.sum()
    U   = np.random.uniform(0, 1.0 / N)
    pts = (np.arange(N) + U) / N
    cdf = np.cumsum(w)
    i, j, idxs = 0, 0, np.zeros(N, dtype=int)
    while i < N:
        if pts[i] <= cdf[j]:
            idxs[i] = j
            i += 1
        else:
            j = min(j + 1, N - 1)
    return idxs

# ── Effective Sample Size ─────────────────────────────────────────────────────
def ess(weights):
    '''ESS = 1 / sum(w_i^2).  Ranges from 1 (total degeneracy) to N (uniform).'''
    w = np.asarray(weights) / np.sum(weights)
    return 1.0 / np.sum(w**2)

print("PF functions defined.")
print("  pf_predict(particles, motion_std)")
print("  pf_reweight(particles, observation, sensor_std)")
print("  systematic_resample(weights)")
print("  ess(weights)")


## Step-by-Step Walkthrough

Let's trace through **one complete iteration** with 200 particles tracking
an object moving along a sine curve.  We show the state *before* and *after*
each operation so you can see exactly what the filter does.


In [ ]:
np.random.seed(42)
N_P = 200                   # number of particles
T   = 60                    # total time steps
motion_std = 0.4            # noise on each step (small = tight tracking)
sensor_std = 0.4            # observation noise

# Object follows y = sin(x/8)*10 + 25, moving +1 in x per step
obj_x = np.linspace(0, T - 1, T)
obj_y = np.sin(obj_x / 8.0) * 10 + 25

# Initialise particles near the starting position
particles = np.column_stack([
    np.random.uniform(-2, 5,  N_P),   # x: start near 0
    np.random.uniform(15, 35, N_P),   # y: cover expected range
])
weights = np.ones(N_P) / N_P

# Store history for animation
hist_particles = [particles.copy()]
hist_weights   = [weights.copy()]
hist_obs       = [(obj_x[0], obj_y[0])]
hist_mean      = [np.average(particles, weights=weights, axis=0)]

for t in range(1, T):
    obs = np.array([obj_x[t], obj_y[t]])

    # 1. Predict: x advances ~1 per step (the transition model knows the drift)
    particles[:, 0] += 1.0 + np.random.randn(N_P) * motion_std
    particles[:, 1] +=       np.random.randn(N_P) * motion_std

    # 2. Reweight
    weights = pf_reweight(particles, obs, sensor_std)

    # 3. Resample
    idxs      = systematic_resample(weights)
    particles = particles[idxs]
    weights   = np.ones(N_P) / N_P

    hist_particles.append(particles.copy())
    hist_weights.append(weights.copy())
    hist_obs.append(tuple(obs))
    hist_mean.append(np.mean(particles, axis=0))

means  = np.array(hist_mean)
errors = np.array([np.linalg.norm(means[t] - [obj_x[t], obj_y[t]])
                   for t in range(T)])

print(f"Run complete. {T} steps, {N_P} particles.")
print(f"Mean tracking error: {np.mean(errors):.2f} units")
print(f"Final error:         {errors[-1]:.2f} units")


In [ ]:
# ── Static snapshot: show 4 time steps side by side ───────────────────────────
snapshots = [5, 15, 30, 55]

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
fig.suptitle('Particle Filter Tracking a Sine Curve', fontsize=14, fontweight='bold')

for ax, t in zip(axes, snapshots):
    px, py = hist_particles[t][:, 0], hist_particles[t][:, 1]
    ax.scatter(px, py, s=12, color='#3fb950', alpha=0.4, label='Particles', zorder=2)
    ax.scatter(*hist_obs[t], s=120, color='#f9826c', zorder=5,
               marker='*', label='Observation')
    ax.scatter(*hist_mean[t], s=100, color='cyan', zorder=4,
               marker='D', label='PF estimate')
    ax.plot(obj_x[:t+1], obj_y[:t+1], color='#f9826c', lw=1.5, alpha=0.5)
    ax.set(xlim=(-5, 65), ylim=(0, 50),
           title=f't = {t}',
           xlabel='X', ylabel='Y' if t == snapshots[0] else '')
    if t == snapshots[0]:
        ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('figures/fig_02b_pf_snapshots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Full animation ────────────────────────────────────────────────────────────
fig, (ax_main, ax_err) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Particle Filter Animation', fontsize=13, fontweight='bold')

ax_main.set(xlim=(-5, 65), ylim=(0, 50),
            xlabel='X', ylabel='Y',
            title='Particle cloud tracking a nonlinear path')
ax_err.set(xlim=(0, T), ylim=(0, None),
           xlabel='Time step', ylabel='Error (L2)',
           title='Tracking error over time')

scat_p  = ax_main.scatter([], [], s=10,  color='#3fb950', alpha=0.35, zorder=2)
scat_o  = ax_main.scatter([], [], s=150, color='#f9826c', marker='*',  zorder=5)
scat_m  = ax_main.scatter([], [], s=100, color='cyan',    marker='D',  zorder=4)
line_t, = ax_main.plot([], [], color='#f9826c', lw=1.5, alpha=0.4)
line_e, = ax_err.plot([], [], color='#f0883e', lw=2)

ax_main.legend(handles=[
    mpatches.Patch(color='#3fb950', alpha=0.5, label='Particles'),
    plt.Line2D([0],[0], marker='*', color='w', markerfacecolor='#f9826c',
               markersize=12, label='Observation'),
    plt.Line2D([0],[0], marker='D', color='w', markerfacecolor='cyan',
               markersize=10, label='PF estimate'),
], fontsize=9)

def animate(t):
    px = hist_particles[t][:, 0]
    py = hist_particles[t][:, 1]
    scat_p.set_offsets(np.c_[px, py])
    scat_o.set_offsets([hist_obs[t]])
    scat_m.set_offsets([hist_mean[t]])
    line_t.set_data(obj_x[:t+1], obj_y[:t+1])
    line_e.set_data(np.arange(t+1), errors[:t+1])
    ax_err.set_ylim(0, max(errors[:t+1].max() * 1.2, 1))
    return scat_p, scat_o, scat_m, line_t, line_e

anim = FuncAnimation(fig, animate, frames=T, interval=80, blit=True)
plt.tight_layout()
HTML(anim.to_jshtml())


## Key Takeaways

| Step | What happens | Intuition |
|------|-------------|-----------|
| **Predict** | Add Gaussian noise to particles | "The world moves; we're not sure how far" |
| **Reweight** | Score by distance to observation | "Particles near the truth are more likely" |
| **Resample** | Duplicate heavy, delete light | "Keep the best hypotheses" |
| **Repeat** | Every time new data arrives | "Continuously correct drift" |

**Key parameters:**
- `motion_std` — how much to spread particles during prediction
- `sensor_std` — how noisy are the observations?
- `N_particles` — more particles = better approximation of the posterior

**Next: Module 3** — we replace the toy 1D motion with a full Agent-Based Model,
so each "particle" is an entire crowd simulation.
